In [ ]:
# @title 1. 원본 의존성 설치 (로그 표시, 자동 종료 없음)
import subprocess
import sys

print("시스템 의존성 설치 시작", flush=True)
subprocess.run(["sudo", "apt", "install", "-y", "tesseract-ocr"], check=True)
print("Python 패키지 설치 시작", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "tensorflow", "tensorflow-text", "pytesseract", "tf-keras",
    "python-doctr[tf,viz]", "huggingface-hub>=1.3.0,<2.0",
    "Pillow", "numpy",
], check=True)
print("설치 완료. 런타임을 수동으로 재시작한 뒤 2번 셀부터 실행하세요.", flush=True)


**여기서 런타임을 수동으로 재시작하세요.Ctrl+M .**
코랩의 런타임 메뉴에서 세션 재시작을 선택하고, 연결이 복구되면 아래 2번 셀부터 실행합니다.
강제 종료 코드는 사용하지 않습니다. 설치 중 오류가 발생했다면 아래 단계로 진행하지 말고 설치 로그를 확인하세요.


# 기본 세팅

In [ ]:
# @title 2. 라이브러리 불러오기 및 환경 확인
import sys
print("Python:", sys.version, flush=True)
print("TensorFlow 불러오는 중...", flush=True)
import tensorflow as tf
print("TensorFlow:", tf.__version__, flush=True)
import tensorflow_text  # SavedModel의 텍스트 연산 등록
print("TensorFlow Text:", tensorflow_text.__version__, flush=True)

import json
import re
import warnings
from pathlib import Path
import numpy as np
from PIL import Image
from huggingface_hub import snapshot_download

print("GPU:", tf.config.list_physical_devices("GPU"), flush=True)
print("라이브러리 준비 완료", flush=True)


In [ ]:
# @title 3. InkSight 모델 다운로드 (원본과 동일한 모델)
print("InkSight-Small-p 다운로드 시작. 첫 실행은 시간이 걸릴 수 있습니다.", flush=True)
model_path = snapshot_download("Derendering/InkSight-Small-p")
print("다운로드 완료:", model_path, flush=True)


In [ ]:
# @title 4. 모델 로드 (런타임당 한 번)
print("SavedModel을 메모리에 불러오는 중...", flush=True)
model = tf.saved_model.load(model_path)
cf = model.signatures["serving_default"]
print("모델 준비 완료", flush=True)


In [ ]:
# @title 5. 획 변환 함수 정의
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

def text_to_tokens(text) -> list[int]:
    pattern = r"<ink_token_(\d+)>"
    matches = re.findall(pattern, text)
    return [int(tok) for tok in matches]

def detokenize(tokens: list[int]):
    coordinate_length = 224
    num_token_per_dimension = coordinate_length + 1
    vocabulary_size = num_token_per_dimension * 2 + 1
    start_token = num_token_per_dimension * 2

    if any([t < 0 or t >= vocabulary_size for t in tokens]):
        raise ValueError(
            f"Ink token indices should be between 0 and {vocabulary_size}"
        )
    idx = 0
    res = []
    current_stroke_tokens = []

    while idx < len(tokens):
        token = tokens[idx]
        if token == start_token:
            if current_stroke_tokens:
                res.append(current_stroke_tokens)
            current_stroke_tokens = []
            idx += 1
        else:
            if idx + 1 < len(tokens) and (tokens[idx + 1] != start_token):
                # Read in x and y coordinates.
                x = tokens[idx]
                y = tokens[idx + 1] - num_token_per_dimension
                # If the coordinates are valid, add them to detokenization ink.
                if (0 <= x <= coordinate_length) and (0 <= y <= coordinate_length):
                    current_stroke_tokens.append([x, y])
                idx += 2
            # If y doesn't exist or y is start_token, then skip this x.
            else:
                idx += 1
    if current_stroke_tokens:
        res.append(current_stroke_tokens)

    class Stroke:
        def __init__(self, list_of_coordinates=None) -> None:
            self.x = []
            self.y = []
            if list_of_coordinates:
                for point in list_of_coordinates:
                    self.x.append(point[0])
                    self.y.append(point[1])

        def __len__(self):
            return len(self.x)

        def __getitem__(self, index):
            return (self.x[index], self.y[index])

    class Ink:
        def __init__(self, list_of_strokes=None) -> None:
            self.strokes = []
            if list_of_strokes:
                self.strokes = list_of_strokes

        def __len__(self):
            return len(self.strokes)

        def __getitem__(self, index):
            return self.strokes[index]

    strokes = []
    for stroke_data in res:
        stroke_points = []
        for point in stroke_data:
            x, y = point
            stroke_points.append((x, y))
        strokes.append(Stroke(stroke_points))
    return Ink(strokes)

def process_image_and_save_json(image_path, output_path):
    """이미지의 획을 JSON 파일로 저장하고 저장 경로(str)를 반환합니다.

    image_path: 코랩에서 접근 가능한 입력 이미지 파일 경로.
    output_path: 파일명과 .json을 포함한 저장 경로. 부모 폴더는 자동 생성합니다.
        기존 파일이 있으면 덮어씁니다.
    JSON 형식: {"strokes": [[{"x": 10, "y": 20}, ...], ...]}
    좌표는 원본과 동일하게 검은 여백을 포함한 224×224 이미지 기준입니다.
    두 프롬프트에서 모두 획이 없으면 {"strokes": []}를 저장합니다.
    모델은 위의 준비 셀에서 로드되며 같은 런타임에서 재사용됩니다.
    """
    image_path = Path(image_path).expanduser()
    output_path = Path(output_path).expanduser()
    if not image_path.is_file():
        raise FileNotFoundError(f"입력 이미지가 없습니다: {image_path}")
    if output_path.suffix.lower() != ".json":
        raise ValueError("output_path는 파일명을 포함한 .json 경로여야 합니다.")
    if image_path.resolve() == output_path.resolve():
        raise ValueError("입력 이미지와 출력 파일 경로가 같을 수 없습니다.")

    with Image.open(image_path) as original:
        image = original.convert("RGB")
    ratio = min(224 / image.width, 224 / image.height)
    resized = image.resize((max(1, int(image.width * ratio)),
                            max(1, int(image.height * ratio))))
    padded = Image.new("RGB", (224, 224), (0, 0, 0))
    padded.paste(resized, ((224 - resized.width) // 2,
                          (224 - resized.height) // 2))

    infer = cf
    encoded = tf.reshape(tf.io.encode_jpeg(np.array(padded)), (1, 1))
    for prompt in ("Recognize and derender.", "Derender the ink."):
        output = infer(**{
            "input_text": tf.constant([prompt], dtype=tf.string),
            "image/encoded": encoded,
        })
        text = output["output_0"].numpy()[0][0].decode("utf-8")
        ink = detokenize(text_to_tokens(text))
        if ink.strokes:
            break
    data = {"strokes": [
        [{"x": x, "y": y} for x, y in zip(stroke.x, stroke.y)]
        for stroke in ink.strokes
    ]}
    if not data["strokes"]:
        warnings.warn(f"획을 검출하지 못했습니다: {image_path}", stacklevel=2)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    return str(output_path)


In [ ]:
#@title 골라내기 함수

import json
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import unicodedata

def _plot_current_image_strokes(img_name, image_path_full, strokes_data, plot_output_area):
    with plot_output_area:
        clear_output(wait=True)
        if not os.path.exists(image_path_full):
            print(f"Error: Image file not found at {image_path_full}. Cannot plot.")
            return
        try:
            with Image.open(image_path_full) as src:
                image = src.convert("RGB")
            ratio = min(224 / image.width, 224 / image.height)
            resized = image.resize((
                max(1, int(image.width * ratio)),
                max(1, int(image.height * ratio)),
            ))
            padded = Image.new("RGB", (224, 224), (0, 0, 0))
            padded.paste(resized, (
                (224 - resized.width) // 2,
                (224 - resized.height) // 2,
            ))
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            axes[0].imshow(padded)
            axes[0].set_title("Input")
            axes[1].imshow(padded, alpha=0.5)
            axes[1].set_title(f"Overlay ({len(strokes_data)} strokes)")
            axes[2].set_title("Strokes only")
            cmap = plt.get_cmap("turbo")
            for i, stroke in enumerate(strokes_data):
                if not stroke:
                    continue
                x = [p["x"] for p in stroke]
                y = [p["y"] for p in stroke]
                color = cmap(i / max(1, len(strokes_data) - 1))
                for ax in axes[1:]:
                    ax.plot(x, y, color=color, linewidth=2, marker=".", markersize=3)
                    ax.annotate(str(i + 1), (x[0], y[0]),
                                xytext=(4, 4), textcoords="offset points",
                                color=color, fontsize=20)
            for ax in axes:
                ax.set_xlim(-0.5, 224.5)
                ax.set_ylim(224.5, -0.5)
                ax.set_aspect("equal")
                ax.axis("off")
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Error plotting image {img_name}: {e}")

def _merge_strokes_logic(strokes_data, stroke_idx1, stroke_idx2):
    s1 = strokes_data[stroke_idx1]
    s2 = strokes_data[stroke_idx2]
    s1_start = s1[0]
    s1_end = s1[-1]
    s2_start = s2[0]
    s2_end = s2[-1]
    endpoints = [
        (s1_start, s2_start, "s1_start_s2_start"),
        (s1_start, s2_end, "s1_start_s2_end"),
        (s1_end, s2_start, "s1_end_s2_start"),
        (s1_end, s2_end, "s1_end_s2_end"),
    ]
    min_dist = float("inf")
    closest_pair_type = ""
    for ep1, ep2, pair_type in endpoints:
        dist = ((ep1["x"] - ep2["x"])**2 + (ep1["y"] - ep2["y"])**2)**0.5
        if dist < min_dist:
            min_dist = dist
            closest_pair_type = pair_type
    merged_stroke = []
    if closest_pair_type == "s1_start_s2_start":
        merged_stroke = list(reversed(s2)) + s1
    elif closest_pair_type == "s1_start_s2_end":
        merged_stroke = s2 + list(reversed(s1))
    elif closest_pair_type == "s1_end_s2_start":
        merged_stroke = s1 + s2
    elif closest_pair_type == "s1_end_s2_end":
        merged_stroke = s1 + list(reversed(s2))
    if stroke_idx1 < stroke_idx2:
        new_strokes_data = [s for i, s in enumerate(strokes_data) if i != stroke_idx1 and i != stroke_idx2]
    else:
        new_strokes_data = [s for i, s in enumerate(strokes_data) if i != stroke_idx2 and i != stroke_idx1]
    new_strokes_data.append(merged_stroke)
    return new_strokes_data

def interactive_stroke_selection(
    image_names_to_process_list,
    image_paths_base_dir,
    json_paths_base_dir,
    session_display_name,
    image_data_lookup_dict,
    global_clean_list,
    current_session_skip_list,
    enable_auto_skip=False
):
    image_index = 0
    plot_output_area = widgets.Output()
    control_output_area = widgets.Output()
    display(plot_output_area, control_output_area)

    def display_reorder_interface(current_img_name, current_image_path_full, current_json_path_full, current_strokes_data):
        _plot_current_image_strokes(current_img_name, current_image_path_full, current_strokes_data, plot_output_area)
        with control_output_area:
            clear_output(wait=True)
            print(f"'{current_img_name}' 획 순서 변경")
            current_order_str = ','.join([str(i+1) for i in range(len(current_strokes_data))])
            order_input = widgets.Text(
                value=current_order_str,
                placeholder='예: 1,3,2,4 또는 1,2 (일부 삭제)',
                description='새 획 순서:',
                disabled=False,
                layout=widgets.Layout(width='auto')
            )
            apply_button = widgets.Button(description="순서 변경 및 재표시", button_style='success')
            cancel_button = widgets.Button(description="취소 (메인 화면으로)", button_style='')

            def on_apply_clicked(b_apply):
                new_order_str = order_input.value
                try:
                    new_order_indices = []
                    if new_order_str.strip():
                        new_order_indices = [int(x.strip()) - 1 for x in new_order_str.split(',') if x.strip()]
                    if not all(0 <= idx < len(current_strokes_data) for idx in new_order_indices):
                        raise ValueError("유효하지 않은 획 번호가 포함되어 있습니다.")
                    if len(new_order_indices) != len(set(new_order_indices)):
                        raise ValueError("획 번호가 중복되어 나타납니다.")
                    reordered_strokes = [current_strokes_data[i] for i in new_order_indices]
                    updated_json_data = {"strokes": reordered_strokes}
                    with open(current_json_path_full, "w", encoding="utf-8") as f:
                        json.dump(updated_json_data, f, ensure_ascii=False, indent=4)
                    with control_output_area:
                        print(f"'{current_img_name}' 획 순서가 임시 적용 및 저장되었습니다.")
                    display_image_and_buttons()
                except ValueError as ve:
                    with control_output_area:
                        print(f"오류: {ve}. 다시 시도해주세요.")
                        display(order_input, widgets.HBox([apply_button, cancel_button]))

            def on_cancel_clicked(b_cancel):
                display_image_and_buttons()

            display(order_input, widgets.HBox([apply_button, cancel_button]))
            apply_button.on_click(on_apply_clicked)
            order_input.on_submit(on_apply_clicked)
            cancel_button.on_click(on_cancel_clicked)

    def display_merge_interface(current_img_name, current_image_path_full, current_json_path_full, current_strokes_data):
        _plot_current_image_strokes(current_img_name, current_image_path_full, current_strokes_data, plot_output_area)
        with control_output_area:
            clear_output(wait=True)
            print(f"'{current_img_name}' 획 합치기")
            merge_input = widgets.Text(
                placeholder='예: 1,2',
                description='합칠 획 번호:',
                disabled=False,
                layout=widgets.Layout(width='auto')
            )
            apply_merge_button = widgets.Button(description="합치기 및 재표시", button_style='success')
            cancel_merge_button = widgets.Button(description="취소 (메인 화면으로)", button_style='')

            def on_merge_apply_clicked(b_apply_merge):
                merge_str = merge_input.value
                try:
                    stroke_indices_to_merge = [int(x.strip()) - 1 for x in merge_str.split(',') if x.strip()]
                    if len(stroke_indices_to_merge) != 2:
                        raise ValueError("정확히 두 개의 획 번호를 입력해야 합니다.")
                    s_idx1, s_idx2 = stroke_indices_to_merge
                    if not (0 <= s_idx1 < len(current_strokes_data) and 0 <= s_idx2 < len(current_strokes_data)):
                        raise ValueError("유효하지 않은 획 번호가 포함되어 있습니다.")
                    if s_idx1 == s_idx2:
                        raise ValueError("같은 획을 두 번 합칠 수 없습니다.")
                    new_strokes_data = _merge_strokes_logic(current_strokes_data, s_idx1, s_idx2)
                    updated_json_data = {"strokes": new_strokes_data}
                    with open(current_json_path_full, "w", encoding="utf-8") as f:
                        json.dump(updated_json_data, f, ensure_ascii=False, indent=4)
                    with control_output_area:
                        print(f"'{current_img_name}' 획이 임시 합쳐지고 저장되었습니다.")
                    display_image_and_buttons()
                except ValueError as ve:
                    with control_output_area:
                        print(f"오류: {ve}. 다시 시도해주세요.")
                        display(merge_input, widgets.HBox([apply_merge_button, cancel_merge_button]))

            def on_merge_cancel_clicked(b_cancel_merge):
                display_image_and_buttons()

            display(merge_input, widgets.HBox([apply_merge_button, cancel_merge_button]))
            apply_merge_button.on_click(on_merge_apply_clicked)
            merge_input.on_submit(on_merge_apply_clicked)
            cancel_merge_button.on_click(on_merge_cancel_clicked)

    def display_image_and_buttons():
        nonlocal image_index
        with control_output_area:
            clear_output(wait=True)

        # 루프를 돌면서 이미 clean이나 skip 리스트에 포함된 문자는 자동으로 인덱스를 증가시키며 스킵합니다.
        while image_index < len(image_names_to_process_list):
            temp_img_name = image_names_to_process_list[image_index]
            norm_temp_name = unicodedata.normalize('NFC', temp_img_name)

            # 전역 리스트 및 현재 세션 스킵 리스트에 존재하면 자동으로 다음 문자 탐색
            is_clean = any(unicodedata.normalize('NFC', c) == norm_temp_name for c in global_clean_list)
            is_skip = any(unicodedata.normalize('NFC', s) == norm_temp_name for s in current_session_skip_list)

            if is_clean or is_skip:
                image_index += 1
            else:
                break

        if image_index >= len(image_names_to_process_list):
            with control_output_area:
                clear_output(wait=True)
                print(f"모든 {session_display_name} 이미지 처리가 완료되었습니다.")
                print(f"Clean 목록: {global_clean_list}")
                print(f"Skip 목록: {current_session_skip_list}")
            with plot_output_area:
                clear_output(wait=True)
            return

        img_name = image_names_to_process_list[image_index]
        expected_stroke_count = image_data_lookup_dict.get(img_name)
        image_path_full = os.path.join(image_paths_base_dir, img_name + '.png')
        json_path_full = os.path.join(json_paths_base_dir, img_name + '.json')
        norm_img_name = unicodedata.normalize('NFC', img_name)

        with control_output_area:
            print(f"\nProcessing {session_display_name} image: {img_name} ({image_index+1}/{len(image_names_to_process_list)})")

        try:
            if not os.path.exists(json_path_full):
                raise FileNotFoundError(f"JSON file not found at {json_path_full}")
            with open(json_path_full, encoding="utf-8") as f:
                strokes = json.load(f)["strokes"]

            if enable_auto_skip and expected_stroke_count is not None:
                current_count = len(strokes)
                if current_count < expected_stroke_count:
                    with control_output_area:
                        print(f"획 수 불일치 자동 스킵: {img_name}")
                    if norm_img_name not in current_session_skip_list:
                        current_session_skip_list.append(norm_img_name)
                    if norm_img_name in global_clean_list:
                        global_clean_list.remove(norm_img_name)
                    image_index += 1
                    display_image_and_buttons()
                    return

            _plot_current_image_strokes(img_name, image_path_full, strokes, plot_output_area)

            clean_button = widgets.Button(description="Clean (승인)", button_style='success')
            skip_button = widgets.Button(description="Skip (건너뛰기)", button_style='info')
            reorder_button = widgets.Button(description="Reorder (순서 변경)", button_style='warning')
            merge_button = widgets.Button(description="Merge (획 합치기)", button_style='danger')

            def on_clean_clicked(b):
                nonlocal image_index
                # 유효성 체크: 획이 전혀 없거나 모든 획이 빈 리스트인 경우 승인 방지
                has_valid_stroke = len(strokes) > 0 and any(len(s) > 0 for s in strokes)
                if not has_valid_stroke:
                    with control_output_area:
                        print("Error: 획이 비어있는 데이터는 Clean으로 승인할 수 없습니다. Skip을 누르거나 순서 수정을 해주세요.")
                    return
                if norm_img_name not in global_clean_list:
                    global_clean_list.append(norm_img_name)
                if norm_img_name in current_session_skip_list:
                    current_session_skip_list.remove(norm_img_name)
                image_index += 1
                display_image_and_buttons()

            def on_skip_clicked(b):
                nonlocal image_index
                if norm_img_name not in current_session_skip_list:
                    current_session_skip_list.append(norm_img_name)
                if norm_img_name in global_clean_list:
                    global_clean_list.remove(norm_img_name)
                image_index += 1
                display_image_and_buttons()

            def on_reorder_clicked(b):
                display_reorder_interface(img_name, image_path_full, json_path_full, strokes)

            def on_merge_clicked(b):
                display_merge_interface(img_name, image_path_full, json_path_full, strokes)

            clean_button.on_click(on_clean_clicked)
            reorder_button.on_click(on_reorder_clicked)
            merge_button.on_click(on_merge_clicked)
            skip_button.on_click(on_skip_clicked)

            with control_output_area:
                display(widgets.HBox([clean_button, skip_button, reorder_button, merge_button]))
        except Exception as e:
            with control_output_area:
                print(f"에러로 인해 스킵: {e}")
            image_index += 1
            display_image_and_buttons()

    display_image_and_buttons()


In [ ]:
#@title stroke_count

# 초성
CHO = [
    'ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ',
    'ㅂ', 'ㅃ', 'ㅅ', 'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ',
    'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ'
]

# 중성
JUNG = [
    'ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ',
    'ㅖ', 'ㅗ', 'ㅘ', 'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ',
    'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ'
]

# 종성
JONG = [
    '', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ',
    'ㄷ', 'ㄹ', 'ㄺ', 'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ',
    'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ', 'ㅆ',
    'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ'
]

stroke = {
    # 자음
    'ㄱ': 1,
    'ㄲ': 2,
    'ㄴ': 1,
    'ㄷ': 2,
    'ㄸ': 4,
    'ㄹ': 3,
    'ㅁ': 3,
    'ㅂ': 4,
    'ㅃ': 8,
    'ㅅ': 2,
    'ㅆ': 4,
    'ㅇ': 1,
    'ㅈ': 2,
    'ㅉ': 4,
    'ㅊ': 3,
    'ㅋ': 2,
    'ㅌ': 3,
    'ㅍ': 4,
    'ㅎ': 3,

    # 모음
    'ㅏ': 2,
    'ㅐ': 3,
    'ㅑ': 3,
    'ㅒ': 4,
    'ㅓ': 2,
    'ㅔ': 3,
    'ㅕ': 3,
    'ㅖ': 4,
    'ㅗ': 2,
    'ㅘ': 4,
    'ㅙ': 5,
    'ㅚ': 3,
    'ㅛ': 3,
    'ㅜ': 2,
    'ㅝ': 4,
    'ㅞ': 5,
    'ㅟ': 3,
    'ㅠ': 3,
    'ㅡ': 1,
    'ㅢ': 2,
    'ㅣ': 1,
}

# 겹받침은 각 자모의 합
stroke.update({
    'ㄳ': stroke['ㄱ'] + stroke['ㅅ'],
    'ㄵ': stroke['ㄴ'] + stroke['ㅈ'],
    'ㄶ': stroke['ㄴ'] + stroke['ㅎ'],
    'ㄺ': stroke['ㄹ'] + stroke['ㄱ'],
    'ㄻ': stroke['ㄹ'] + stroke['ㅁ'],
    'ㄼ': stroke['ㄹ'] + stroke['ㅂ'],
    'ㄽ': stroke['ㄹ'] + stroke['ㅅ'],
    'ㄾ': stroke['ㄹ'] + stroke['ㅌ'],
    'ㄿ': stroke['ㄹ'] + stroke['ㅍ'],
    'ㅀ': stroke['ㄹ'] + stroke['ㅎ'],
    'ㅄ': stroke['ㅂ'] + stroke['ㅅ'],
})


def stroke_count(char):
    if len(char) != 1:
        raise ValueError("한 글자만 입력해야 합니다.")

    code = ord(char)

    # 완성형 한글 범위
    if not 0xAC00 <= code <= 0xD7A3:
        raise ValueError("완성형 한글 한 글자를 입력해야 합니다.")

    index = code - 0xAC00

    cho_idx = index // (21 * 28)
    jung_idx = (index % (21 * 28)) // 28
    jong_idx = index % 28

    cho = CHO[cho_idx]
    jung = JUNG[jung_idx]
    jong = JONG[jong_idx]

    return stroke[cho] + stroke[jung] + (stroke[jong] if jong else 0)

# 타겟 공장

In [ ]:
# @title 사전 세팅
from google.colab import drive
import os
import pandas as pd
import unicodedata
drive.mount('/content/drive')

ttfname='NanumBarunpenB'
ttf="/content/drive/Shareddrives/2026 자율연구/ttfs and zips/"+ttfname+".ttf"

base_path = "/content/results/"
test=True

target_path=base_path+"target/"
output=base_path
csv_path='/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/'+('test_dataset' if test else 'automated_train_dataset')+'/path.csv'
bjson_path='/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/'+('test_dataset' if test else 'automated_train_dataset')+'/jsons/'+ttfname+'/'
stat_path='/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/'+('test_dataset' if test else 'automated_train_dataset')+'/status/status'+ttfname+'.csv'

chars='초빙나눔펜'

completed_chars = set()
if os.path.exists(stat_path):
    try:
        df_stat = pd.read_csv(stat_path)
        if 'done' in df_stat.columns:
            for c in df_stat['done'].dropna().astype(str):
                completed_chars.add(unicodedata.normalize('NFC', c))
        if 'skip' in df_stat.columns:
            for c in df_stat['skip'].dropna().astype(str):
                completed_chars.add(unicodedata.normalize('NFC', c))
        print(f"상태 파일 로드 완료. 필터링 대상 글자 수: {len(completed_chars)}개")
    except Exception as e:
        print(f"상태 파일을 읽는 도중 오류 발생 (무시하고 진행): {e}")

char=dict()
filtered_chars = []
for i in chars:
    normalized_char = unicodedata.normalize('NFC', i)
    if normalized_char not in completed_chars:
        char[i]=stroke_count(i)
    else:
        filtered_chars.append(i)

if filtered_chars:
    print(f"이미 완료/스킵 처리되어 제외된 글자: {filtered_chars}")

# 검수 시작 시점의 전체 글자 목록을 완벽하게 보존
initial_work_chars = [unicodedata.normalize('NFC', k) for k in char.keys()]
print(f"검수 시작 시점 원본 작업 대상: {initial_work_chars}")

count_per_char=5
stroke_thickness=4
max_angle=10
max_factor=0.07

print(f"새로 작업할 글자 수: {len(char)} ({char})")

In [ ]:
# @title 'char' 리스트의 각 문자에 대한 기본 이미지 생성
from PIL import Image, ImageDraw, ImageFont
import os
import shutil

# 'char', 'ttf', 'output' 변수가 이전 셀에서 정의되었다고 가정합니다.

output_char_dir = os.path.join(output, 'target')
os.makedirs(output_char_dir, exist_ok=True)

image_size = 224 # 최종 이미지 크기는 224x224
background_color = (0, 0, 0) # 검은색 배경
text_color = (255, 255, 255) # 흰색 글자

# 참조 스크립트에서 가져온 파라미터 ('0.png' 스타일 매칭)
TARGET_GLYPH_WIDTH = 142
TARGET_GLYPH_HEIGHT = 166
REFERENCE_CONTENT_SIZE = (TARGET_GLYPH_WIDTH, TARGET_GLYPH_HEIGHT)
REFERENCE_CONTENT_BOX = (42, 29, 184, 195) # (left, top, right, bottom) 글리프가 붙여넣기 될 위치

# image_data와 clean 리스트를 초기화하거나 기존 값을 사용
if 'image_data' not in globals():
    image_data = {}
if 'clean' not in globals():
    clean = []

for char_to_generate in char.keys():
    char_image_filename = f"{char_to_generate}.png"
    char_image_path = os.path.join(output_char_dir, ttfname+'_'+char_image_filename)

    # --- 참조 스크립트 기반 이미지 생성 로직 시작 ---

    # 1. REFERENCE_CONTENT_SIZE에 가장 잘 맞는 폰트 크기 선택
    optimal_font = None
    best_error = float("inf")
    # 폰트 크기 후보를 넓은 범위(1부터 200)로 탐색합니다.
    for font_size_candidate in range(1, 201):
        try:
            current_font = ImageFont.truetype(ttf, font_size_candidate)
        except IOError:
            continue # 폰트를 로드할 수 없으면 건너뜁니다.

        # 현재 폰트 크기로 렌더링된 글리프의 바운딩 박스 얻기
        bbox = current_font.getbbox(char_to_generate)
        rendered_width = bbox[2] - bbox[0]
        rendered_height = bbox[3] - bbox[1]

        # REFERENCE_CONTENT_SIZE와 렌더링된 크기 간의 오차 제곱합 계산
        error = (rendered_width - TARGET_GLYPH_WIDTH)**2 + (rendered_height - TARGET_GLYPH_HEIGHT)**2

        if error < best_error:
            best_error = error
            optimal_font = current_font

    if optimal_font is None:
        # 최적 폰트를 찾지 못하면 기본 폰트로 폴백
        print(f"경고: '{char_to_generate}'에 대한 최적 폰트 크기를 찾을 수 없습니다. 기본 폰트를 사용합니다.")
        optimal_font = ImageFont.load_default()
        # 기본 폰트 사용 시에는 REFERENCE_CONTENT_SIZE 로직을 따르지 않고 일반적인 중앙 정렬 로직을 사용해야 하지만,
        # 사용자 요청의 핵심은 참조 파라미터 적용이므로, 이 경우 오류를 명확히 하고 다음 단계로 진행합니다.
        # ttf 변수에 올바른 폰트 경로가 설정되어 있다면 이 경고는 발생하지 않아야 합니다.


    # 2. 임시 'L' (그레이스케일) 이미지에 글리프 렌더링
    # 선택된 최적 폰트로 글리프의 바운딩 박스 다시 계산
    bbox_optimal = optimal_font.getbbox(char_to_generate)
    rendered_content_width = bbox_optimal[2] - bbox_optimal[0]
    rendered_content_height = bbox_optimal[3] - bbox_optimal[1]

    # 글리프를 그릴 임시 그레이스케일 이미지 생성
    glyph_temp_image = Image.new("L", (rendered_content_width, rendered_content_height), 0)
    glyph_draw = ImageDraw.Draw(glyph_temp_image)

    # 임시 이미지에 문자 그리기 (fill=255로 흰색 글리프)
    # 바운딩 박스의 음수 오프셋을 고려하여 (0,0)에 시작하도록 조정
    glyph_draw.text((-bbox_optimal[0], -bbox_optimal[1]), char_to_generate, font=optimal_font, fill=255)

    # 글리프 이미지의 실제 내용만 포함하도록 이미지 자르기
    try:
        glyph_cropped = glyph_temp_image.crop(glyph_temp_image.getbbox())
    except ValueError: # getbbox()가 비어있는 이미지를 반환할 경우를 대비
        glyph_cropped = glyph_temp_image

    # 3. 잘라낸 글리프를 REFERENCE_CONTENT_SIZE (142x166)로 정확히 크기 조정
    glyph_resized = glyph_cropped.resize(REFERENCE_CONTENT_SIZE, Image.Resampling.LANCZOS)

    # 4. 검은색 배경의 최종 224x224 이미지 생성
    final_image = Image.new("RGB", (image_size, image_size), background_color)

    # 5. 크기 조정된 글리프를 final_image의 REFERENCE_CONTENT_BOX 위치에 붙여넣기
    # paste 함수는 (대상 이미지 내의 left, top) 좌표와 마스크를 사용합니다.
    # text_color를 붙여넣기 색상으로 사용하고, glyph_resized를 마스크로 사용하여 글리프만 흰색으로 표시
    final_image.paste(text_color, REFERENCE_CONTENT_BOX, glyph_resized)

    # --- 사용자 정의 이미지 생성 로직 끝 ---

    # 최종 이미지 저장
    final_image.save(char_image_path)
    print(f"문자 '{char_to_generate}' 이미지를 '{char_image_path}'에 저장했습니다.")

    # image_data와 clean 리스트 업데이트
    generated_image_key = f"generated_char_{char_to_generate}"
    image_data[generated_image_key] = 2 # 획 수는 이미지에서 직접 추출되지 않으므로 예상 값으로 설정
    if generated_image_key not in clean:
        clean.append(generated_image_key)

print("모든 'char' 문자에 대한 이미지 생성 및 리스트 업데이트 완료.")

In [ ]:
import os
# @title json 생성 for target
extracted=[]
# image_data 딕셔너리의 키(이미지 이름)를 순회합니다.
for image_name in char.keys():
    try:
        target_json_path = bjson_path+image_name + '.json'
        if os.path.exists(target_json_path):
            print(f"{image_name}.json 이 이미 존재하여 건너뜁니다.")
            extracted.append(image_name)
            continue

        process_image_and_save_json(
            image_path=target_path+ttfname+'_'+image_name+'.png',
            output_path=target_json_path
        )
        extracted.append(image_name);
        print(image_name, len(extracted),'/',len(char))
    except Exception as e: # Catch specific exceptions for better error handling
        print(f"{image_name}가 추출되지 못함: {e}")

print('총 '+str(len(extracted))+'개 추출 완료')

In [ ]:
# 'target' 이미지에 대한 'clean' 및 'skip' 리스트 사전세팅
#local_target_clean = ['NanumDaSiSiJagHae_아', 'NanumDaSiSiJagHae_야', 'NanumDaSiSiJagHae_강', 'NanumDaSiSiJagHae_냠', 'NanumDaSiSiJagHae_떡', 'NanumDaSiSiJagHae_련', 'NanumDaSiSiJagHae_삯', 'NanumDaSiSiJagHae_쌀', 'NanumDaSiSiJagHae_쩐', 'NanumDaSiSiJagHae_첸', 'NanumDaSiSiJagHae_옛', 'NanumDaSiSiJagHae_혀', 'NanumDaSiSiJagHae_공', 'NanumDaSiSiJagHae_꾀', 'NanumDaSiSiJagHae_뇌', 'NanumDaSiSiJagHae_돌', 'NanumDaSiSiJagHae_똥', 'NanumDaSiSiJagHae_몫', 'NanumDaSiSiJagHae_뜸', 'NanumDaSiSiJagHae_찜', 'NanumDaSiSiJagHae_콩', 'NanumDaSiSiJagHae_흉', 'NanumDaSiSiJagHae_궤', 'NanumDaSiSiJagHae_된', 'NanumDaSiSiJagHae_뒨', 'NanumDaSiSiJagHae_쉰', 'NanumDaSiSiJagHae_쥔', 'NanumDaSiSiJagHae_튄', 'NanumDaSiSiJagHae_희', 'NanumDaSiSiJagHae_늬']
#local_target_skip = ['NanumDaSiSiJagHae_깨', 'NanumDaSiSiJagHae_댁', 'NanumDaSiSiJagHae_멸', 'NanumDaSiSiJagHae_볕', 'NanumDaSiSiJagHae_뼈', 'NanumDaSiSiJagHae_얘', 'NanumDaSiSiJagHae_절', 'NanumDaSiSiJagHae_캔', 'NanumDaSiSiJagHae_털', 'NanumDaSiSiJagHae_괄', 'NanumDaSiSiJagHae_봄', 'NanumDaSiSiJagHae_솥', 'NanumDaSiSiJagHae_쑥', 'NanumDaSiSiJagHae_용', 'NanumDaSiSiJagHae_줌', 'NanumDaSiSiJagHae_춤', 'NanumDaSiSiJagHae_통', 'NanumDaSiSiJagHae_품', 'NanumDaSiSiJagHae_권', 'NanumDaSiSiJagHae_귤', 'NanumDaSiSiJagHae_괜', 'NanumDaSiSiJagHae_윌']
local_target_clean =[]
local_target_skip = []

In [ ]:
# @title 골라내기 for target

try:
    local_target_clean
except NameError:
    local_target_clean = []

try:
    local_target_skip
except NameError:
    local_target_skip=[]

# 검수 완료 여부 게이트 초기화
inspection_completed = False

# 실제 생성된 파일명인 [ttfname]_[글자] 리스트를 구성하여 검수 함수에 넘깁니다.
target_image_names = [f"{ttfname}_{k}" for k in char.keys()]

# 내부 검수용 lookup 딕셔너리도 [ttfname]_[글자] 형식에 맞춰 생성합니다.
target_char_lookup = {f"{ttfname}_{k}": v for k, v in char.items()}

import os
import shutil
import unicodedata

# bjson_path 내부의 실제 파일명들을 NFC로 정규화 매핑하여 탐색
real_json_files = os.listdir(bjson_path) if os.path.exists(bjson_path) else []
json_lookup = {}
for f in real_json_files:
    if f.endswith('.json'):
        norm_name = unicodedata.normalize('NFC', f)
        json_lookup[norm_name] = f

for k in char.keys():
    norm_target_json_filename = f"{k}.json"
    norm_prefixed_json_filename = f"{ttfname}_{k}.json"

    # 원본 글자명(예: 아.json)이 실제 디렉토리에 NFC/NFD 중 하나로 존재하는지 확인
    real_filename = json_lookup.get(unicodedata.normalize('NFC', norm_target_json_filename))

    if real_filename:
        src_json = os.path.join(bjson_path, real_filename)
        # 대화형 툴이 찾을 접두사 포함 파일명(예: NanumDaSiSiJagHae_아.json)으로 복사 또는 링크
        dst_json_nfc = os.path.join(bjson_path, norm_prefixed_json_filename)
        dst_json_nfd = os.path.join(bjson_path, unicodedata.normalize('NFD', norm_prefixed_json_filename))

        for dst in (dst_json_nfc, dst_json_nfd):
            if not os.path.exists(dst):
                try:
                    os.symlink(src_json, dst)
                except Exception:
                    try:
                        shutil.copy(src_json, dst)
                    except Exception as e:
                        pass

interactive_stroke_selection(
    image_names_to_process_list=target_image_names,
    image_paths_base_dir=target_path,
    json_paths_base_dir=bjson_path,
    session_display_name="Target",
    image_data_lookup_dict=target_char_lookup,
    global_clean_list=local_target_clean,
    current_session_skip_list=local_target_skip,
    enable_auto_skip=True
)

In [ ]:
skipmore=['push_빙']

for i in skipmore:
    local_target_clean.remove(ttfname+'_'+i)
    local_target_skip.append(ttfname+'_'+i)

In [ ]:
# @title 검수 완료 여부 최종 검증 및 char 사전 최종 갱신
import unicodedata

# NFC 정규화 완료된 타겟 리스트 생성
normalized_clean_set = {unicodedata.normalize('NFC', k) for k in local_target_clean}
normalized_skip_set = {unicodedata.normalize('NFC', k) for k in local_target_skip}

unresolved_chars = []
for k in initial_work_chars:
    # 검수 대상 이름 형식인 [ttfname]_[글자]로 정규화 매칭 확인
    target_name = f"{ttfname}_{k}"
    normalized_target_name = unicodedata.normalize('NFC', target_name)
    if normalized_target_name not in normalized_clean_set and normalized_target_name not in normalized_skip_set:
        unresolved_chars.append(k)

if unresolved_chars:
    inspection_completed = False
    raise ValueError(f"검증 실패: 아직 완료되지 않은 글자가 존재합니다: {unresolved_chars}. 모든 글자를 Clean 또는 Skip으로 확정한 후 진행하세요.")
else:
    inspection_completed = True
    print("🎉 [검증 통과] 모든 대상 글자의 검수가 정상적으로 완료되었습니다!")

    # char 사전 갱신: 오직 clean 승인된 글자만 남김
    char_keys = list(char.keys())
    for key in char_keys:
        target_name = f"{ttfname}_{key}"
        norm_target_name = unicodedata.normalize('NFC', target_name)
        if norm_target_name not in normalized_clean_set:
            char.pop(key, None)

    print(f"최종 진행 대상 글자 수: {len(char)}개")
    print("최종 진행 대상 목록:", list(char.keys()))

    print('\nlocal_target_clean=',local_target_clean)
    print('local_target_clean=',local_target_skip)

# 공장

In [ ]:
# @title 획 이미지 생성 for target
import os
import shutil
from PIL import Image, ImageDraw
import json
import matplotlib.pyplot as plt
import numpy as np
import re # Import re for filename sanitization

# Control variable for combined image generation
combined_needed = True # Set to False if you don't want to generate combined images

# Define output directories
target_dir = os.path.join(output, 'target')
strokes_dir = os.path.join(output, 'target_stroke')

# Create directories if they don't exist
os.makedirs(target_dir, exist_ok=True)
os.makedirs(strokes_dir, exist_ok=True)

print(f"대상 이미지 저장 디렉토리: {target_dir}")
print(f"획 이미지 저장 디렉토리: {strokes_dir}")

# Helper function to sanitize filenames
def make_safe_filename(original_filename):
    # Replace any character that is NOT a letter, number, underscore, hyphen, or dot with an underscore
    return re.sub(r'[^\w.-]', '_', original_filename)

# Function to get distinct colors for strokes
def get_colors(num_strokes):
    cmap = plt.get_cmap("hsv") # You can choose other colormaps like "turbo", "rainbow", etc.
    colors = []
    if num_strokes == 0:
        return []
    for i in range(num_strokes):
        # Convert 0-1 float color to 0-255 RGB tuple
        r, g, b, _ = cmap(i / num_strokes) # Changed denominator from max(1, num_strokes - 1) to num_strokes
        colors.append((int(r*255), int(g*255), int(b*255)))
    return colors

# Loop through each target image
for img_name in char.keys():
    safe_img_name = make_safe_filename(img_name) # Sanitize the image name for file operations

    original_image_path = os.path.join(target_dir, ttfname+'_'+img_name + '.png') # Use target_dir here
    json_path = os.path.join(bjson_path, img_name + '.json')

    if not os.path.exists(original_image_path):
        print(f"원본 이미지 없음: {original_image_path}. 건너뜁니다.")
        continue
    if not os.path.exists(json_path):
        print(f"JSON 파일 없음: {json_path}. 건너뜁니다.")
        continue

    print(f"'{img_name}' 처리 중...")

    # Original image copy logic is commented out as per user's request.
    # No longer copying original images to target_dir in this cell.

    # Load stroke data
    with open(json_path, 'r', encoding='utf-8') as f:
        stroke_data = json.load(f)["strokes"]

    # 2. Create and save individual stroke images
    for i, stroke_points in enumerate(stroke_data):
        if not stroke_points: # Skip empty strokes if any
            continue

        # Create a blank image with black background (224x224 as per model padding)
        single_stroke_image = Image.new('RGB', (224, 224), color = 'black')
        draw = ImageDraw.Draw(single_stroke_image)

        # Convert stroke points to (x, y) tuples for ImageDraw.line
        line_points = [(p['x'], p['y']) for p in stroke_points]
        if len(line_points) > 1:
            # Draw the stroke in white with specified thickness
            draw.line(line_points, fill='white', width=stroke_thickness)

        individual_stroke_output_path = os.path.join(strokes_dir, f"{ttfname}_{safe_img_name}_{i+1}.png")
        single_stroke_image.save(individual_stroke_output_path)
        # print(f"  -> 개별 획 이미지 '{safe_img_name}_{i+1}.png'를 '{strokes_dir}'에 저장했습니다.")

    # 3. Create and save combined stroke image with distinct colors, if combined_needed is True
    if combined_needed:
        combined_stroke_image = Image.new('RGB', (224, 224), color = 'black') # Black background
        draw_combined = ImageDraw.Draw(combined_stroke_image)

        num_strokes = len(stroke_data)
        colors = get_colors(num_strokes)

        for i, stroke_points in enumerate(stroke_data):
            if not stroke_points:
                continue
            line_points = [(p['x'], p['y']) for p in stroke_points]
            if len(line_points) > 1:
                # Draw each stroke with a distinct color and specified thickness
                draw_combined.line(line_points, fill=colors[i], width=stroke_thickness)

        combined_stroke_output_path = os.path.join(strokes_dir, f"{ttfname}_{safe_img_name}_c.png")
        combined_stroke_image.save(combined_stroke_output_path)
        print(f"  -> 모든 획 결합 이미지 '{ttfname}_{safe_img_name}_c.png'를 '{strokes_dir}'에 저장했습니다.")

print("\n모든 이미지에 대한 획 이미지 생성이 완료되었습니다.")

In [ ]:
#@title 변형시키기
import random
from PIL import Image
import os
import json
import math
import numpy as np

if not globals().get('inspection_completed', False):
    #raise RuntimeError("경고: 검수가 아직 완료되지 않았거나 승인되지 않았습니다. 이전 단계의 검증을 먼저 성공적으로 마쳐야 합니다.")
    pass

output_input_dir = os.path.join(output, 'input')
os.makedirs(output_input_dir, exist_ok=True)
output_json_dir = bjson_path
os.makedirs(output_json_dir, exist_ok=True)

print(f"증강된 이미지 저장 디렉토리: {output_input_dir}")

for char_to_augment in char.keys():
    base_image_path = os.path.join(output, 'target', f"{ttfname}_{char_to_augment}.png")

    # [수정] 골라내기(검수)에서 수정된 '{ttfname}_{글자}.json'을 우선 탐색하고, 없을 경우 원본 '{글자}.json'을 사용하도록 변경합니다.
    inspected_json_path = os.path.join(bjson_path, f"{ttfname}_{char_to_augment}.json")
    original_json_path = os.path.join(bjson_path, f"{char_to_augment}.json")

    if os.path.exists(inspected_json_path):
        base_json_path = inspected_json_path
        print(f"'{char_to_augment}': 검수 완료된 JSON 적용 ({os.path.basename(inspected_json_path)})")
    else:
        base_json_path = original_json_path
        print(f"'{char_to_augment}': 원본 JSON 적용 ({os.path.basename(original_json_path)})")

    if not os.path.exists(base_image_path):
        print(f"경고: 기본 이미지 '{base_image_path}'를 찾을 수 없습니다. 건너뜁니다.")
        continue
    if not os.path.exists(base_json_path):
        print(f"경고: 기본 JSON '{base_json_path}'를 찾을 수 없습니다. 건너뜁니다.")
        continue

    base_image = Image.open(base_image_path).convert("RGB")
    with open(base_json_path, 'r', encoding='utf-8') as f:
        base_json_data = json.load(f)

    for i in range(count_per_char):
        angle = random.uniform(-max_angle, max_angle)
        factor = random.uniform(1 - max_factor, 1 + max_factor)

        # 캔버스 크기 224x224, 중심 좌표 (112, 112)
        cx, cy = 112.0, 112.0
        rad = math.radians(angle)
        cos_a = math.cos(rad)
        sin_a = math.sin(rad)

        # 아핀 순방향 변환 행렬 구성
        a0 = factor * cos_a
        a1 = -factor * sin_a
        a2 = cx - factor * cos_a * cx + factor * sin_a * cy

        b0 = factor * sin_a
        b1 = factor * cos_a
        b2 = cy - factor * sin_a * cx - factor * cos_a * cy

        # Pillow.Image.transform 역행렬 계산
        det = a0 * b1 - a1 * b0
        inv_a0 = b1 / det
        inv_a1 = -a1 / det
        inv_a2 = (a1 * b2 - a2 * b1) / det
        inv_b0 = -b0 / det
        inv_b1 = a0 / det
        inv_b2 = (a2 * b0 - a0 * b2) / det

        final_canvas = base_image.transform(
            (224, 224),
            Image.Transform.AFFINE,
            (inv_a0, inv_a1, inv_a2, inv_b0, inv_b1, inv_b2),
            resample=Image.Resampling.BICUBIC
        )

        augmented_filename = f"I_{ttfname}_{char_to_augment}_{i+1}.png"
        augmented_filepath = os.path.join(output_input_dir, augmented_filename)
        final_canvas.save(augmented_filepath)

        # JSON 획 좌표 정밀하게 변환
        transformed_strokes = []
        for stroke in base_json_data.get("strokes", []):
            transformed_stroke = []
            for pt in stroke:
                tx = a0 * pt['x'] + a1 * pt['y'] + a2
                ty = b0 * pt['x'] + b1 * pt['y'] + b2
                transformed_stroke.append({"x": tx, "y": ty})
            transformed_strokes.append(transformed_stroke)

        # 이미지-좌표 정렬 일치 검증 코드
        if i == 0:
            test_x, test_y = 112.0, 112.0
            mapped_x = a0 * test_x + a1 * test_y + a2
            mapped_y = b0 * test_x + b1 * test_y + b2
            back_x = inv_a0 * mapped_x + inv_a1 * mapped_y + inv_a2
            back_y = inv_b0 * mapped_x + inv_b1 * mapped_y + inv_b2
            assert abs(test_x - back_x) < 1e-5 and abs(test_y - back_y) < 1e-5, "Affine Transformation inverse projection mismatch!"

        augmented_json_data = {"strokes": transformed_strokes}
        augmented_json_filename = f"I_{char_to_augment}_{i+1}.json"
        augmented_json_filepath = os.path.join(output_json_dir, augmented_json_filename)

        with open(augmented_json_filepath, 'w', encoding='utf-8') as f:
            json.dump(augmented_json_data, f, ensure_ascii=False, indent=4)

print("모든 문자에 대한 증강 이미지 및 JSON 생성이 완벽히 정렬 완료되었습니다.")

In [ ]:
# @title 변형시킨거 한번 보기

import random
import os
import json
import matplotlib.pyplot as plt
import re
from PIL import Image

# 실제 생성된 증강 파일 중 하나를 선택합니다.
augmented_images = [f for f in os.listdir(output_input_dir) if f.startswith('I_') and f.endswith('.png')]

if augmented_images:
    # 1. 무작위로 하나의 이미지 선택
    chosen_img_name = random.choice(augmented_images)
    chosen_base_name = os.path.splitext(chosen_img_name)[0] # 예: I_NanumDaSiSiJagHae_야_2

    chosen_img_path = os.path.join(output_input_dir, chosen_img_name)

    # 이미지 이름 패턴 파싱: I_{ttfname}_{char}_{idx} -> JSON 파일은 I_{char}_{idx}.json 형식
    # ttfname에 언더바가 들어갈 수 있으므로 뒤에서부터 매칭
    match = re.match(r'^I_(.+)_([^_]+)_(\d+)$', chosen_base_name)
    if match:
        _, char_name, idx = match.groups()
        expected_json_name = f"I_{char_name}_{idx}.json"
    else:
        # 매칭이 실패할 경우 기본 파일명 규칙 대비 폴백
        expected_json_name = f"{chosen_base_name}.json"

    chosen_json_path = os.path.join(output_json_dir, expected_json_name)

    if os.path.exists(chosen_json_path):
        # 2. 이미지 및 JSON 데이터 로드
        img = Image.open(chosen_img_path)
        with open(chosen_json_path, 'r', encoding='utf-8') as f:
            stroke_data = json.load(f).get("strokes", [])

        # 3. 시각화
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(img)

        # 각 획을 개별 색상으로 그리기
        cmap = plt.get_cmap("turbo")
        num_strokes = len(stroke_data)

        for i, stroke in enumerate(stroke_data):
            if not stroke:
                continue
            xs = [pt['x'] for pt in stroke]
            ys = [pt['y'] for pt in stroke]
            color = cmap(i / max(1, num_strokes - 1))

            # 획 선 및 마커 표시
            ax.plot(xs, ys, color=color, linewidth=3, marker='o', markersize=4, label=f"Stroke {i+1}")

        ax.set_title(f"Overlay Validation: {chosen_base_name}", fontsize=14)
        ax.axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print(f"오류: 선택한 이미지에 대응하는 JSON 파일이 없습니다: {chosen_json_path}")
else:
    print("경고: 증강 이미지 디렉토리에 'I_'로 시작하는 이미지가 존재하지 않습니다.")

In [ ]:
#@title 획 이미지 만들기
import os
import shutil
from PIL import Image, ImageDraw
import json
import matplotlib.pyplot as plt
import numpy as np
import re # Import re for filename sanitization

# Control variable for combined image generation
combined_needed = True # Set to False if you don't want to generate combined images

# Define output directories
strokes_dir = os.path.join(output, 'input_stroke')

# Create directories if they don't exist
os.makedirs(strokes_dir, exist_ok=True)

print(f"획 이미지 저장 디렉토리: {strokes_dir}")

# Helper function to sanitize filenames
def make_safe_filename(original_filename):
    # Replace any character that is NOT a letter, number, underscore, hyphen, or dot with an underscore
    return re.sub(r'[^\w.-]', '_', original_filename)

# Function to get distinct colors for strokes
def get_colors(num_strokes):
    cmap = plt.get_cmap("hsv")
    colors = []
    if num_strokes == 0:
        return []
    for i in range(num_strokes):
        # Convert 0-1 float color to 0-255 RGB tuple
        r, g, b, _ = cmap(i / num_strokes)
        colors.append((int(r*255), int(g*255), int(b*255)))
    return colors

# Get the list of actual augmented images (starting with 'I_') from output_input_dir (/content/results/input)
augmented_files = [os.path.splitext(f)[0] for f in os.listdir(output_input_dir) if f.startswith('I_') and f.endswith('.png')]

# Loop through each augmented image
for img_name in augmented_files:
    safe_img_name = make_safe_filename(img_name) # Sanitize the image name for file operations

    # Correct directory mapping to /content/results/input
    original_image_path = os.path.join(output_input_dir, img_name + '.png')

    # I_{ttfname}_{char}_{idx} 구조에서 원래의 I_{char}_{idx}.json 파일 경로를 유추
    match = re.match(r'^I_(.+)_([^_]+)_(\d+)$', img_name)
    if match:
        _, char_name, idx = match.groups()
        json_filename = f"I_{char_name}_{idx}.json"
    else:
        json_filename = img_name + '.json'

    json_path = os.path.join(bjson_path, json_filename)

    if not os.path.exists(original_image_path):
        print(f"원본 이미지 없음: {original_image_path}. 건너뜁니다.")
        continue
    if not os.path.exists(json_path):
        print(f"JSON 파일 없음: {json_path}. 건너뜁니다.")
        continue

    print(f"'{img_name}' 처리 중...")

    # Load stroke data
    with open(json_path, 'r', encoding='utf-8') as f:
        stroke_data = json.load(f).get("strokes", [])

    # 1. Create and save individual stroke images
    for i, stroke_points in enumerate(stroke_data):
        if not stroke_points: # Skip empty strokes if any
            continue

        # Create a blank image with black background (224x224 as per model padding)
        single_stroke_image = Image.new('RGB', (224, 224), color = 'black')
        draw = ImageDraw.Draw(single_stroke_image)

        # Convert stroke points to (x, y) tuples for ImageDraw.line
        line_points = [(p['x'], p['y']) for p in stroke_points]
        if len(line_points) > 1:
            # Draw the stroke in white with specified thickness
            draw.line(line_points, fill='white', width=stroke_thickness)

        individual_stroke_output_path = os.path.join(strokes_dir, f"{safe_img_name}_{i+1}.png")
        single_stroke_image.save(individual_stroke_output_path)

    # 2. Create and save combined stroke image with distinct colors, if combined_needed is True
    if combined_needed:
        combined_stroke_image = Image.new('RGB', (224, 224), color = 'black') # Black background
        draw_combined = ImageDraw.Draw(combined_stroke_image)

        num_strokes = len(stroke_data)
        colors = get_colors(num_strokes)

        for i, stroke_points in enumerate(stroke_data):
            if not stroke_points:
                continue
            line_points = [(p['x'], p['y']) for p in stroke_points]
            if len(line_points) > 1:
                # Draw each stroke with a distinct color and specified thickness
                draw_combined.line(line_points, fill=colors[i], width=stroke_thickness)

        combined_stroke_output_path = os.path.join(strokes_dir, f"{safe_img_name}_c.png")
        combined_stroke_image.save(combined_stroke_output_path)
        print(f"  -> 모든 획 결합 이미지 '{safe_img_name}_c.png'를 '{strokes_dir}'에 저장했습니다.")

print("\n모든 이미지에 대한 획 이미지 생성이 완료되었습니다.")

In [ ]:
#@title csv 만들기

import pandas as pd
import os
import re
from collections import defaultdict

rows = []

# 명확한 디렉토리 구분 설정
input_dir = os.path.join(output, 'input')
input_strokes_dir = os.path.join(output, 'input_stroke')
target_strokes_dir = os.path.join(output, 'target_stroke')

# 1. 파일 목록 읽어오기
all_input_files = os.listdir(input_dir) if os.path.exists(input_dir) else []
input_images = sorted([
    os.path.splitext(f)[0]
    for f in all_input_files
    if f.startswith('I_') and f.endswith('.png')
])

all_stroke_files = os.listdir(input_strokes_dir) if os.path.exists(input_strokes_dir) else []
all_target_stroke_files = os.listdir(target_strokes_dir) if os.path.exists(target_strokes_dir) else []

# 2. 파일 목록을 딕셔너리로 사전 그룹화하여 탐색 속도 극대화
input_strokes_dict = defaultdict(list)
for f in all_stroke_files:
    if f.endswith('.png') and not f.endswith('_c.png'):
        # 예: I_NanumDaSiSiJagHae_가_1_3.png -> 키: I_NanumDaSiSiJagHae_가_1, 값: I_NanumDaSiSiJagHae_가_1_3.png
        match = re.match(r'^(I_.+)_(\d+)\.png$', f)
        if match:
            base_name, idx = match.groups()
            input_strokes_dict[base_name].append((int(idx), f))

target_strokes_dict = defaultdict(list)
for f in all_target_stroke_files:
    if f.endswith('.png') and not f.endswith('_c.png'):
        # 예: NanumDaSiSiJagHae_가_2.png -> 키: NanumDaSiSiJagHae_가, 값: NanumDaSiSiJagHae_가_2.png
        match = re.match(r'^(.+)_(\d+)\.png$', f)
        if match:
            base_name, idx = match.groups()
            target_strokes_dict[base_name].append((int(idx), f))

# 3. 데이터프레임 행 구성
for img_name in input_images:
    safe_img_name = make_safe_filename(img_name)

    # 정규식 패턴: I_{ttfname}_{char}_{idx} 구조에서 char_name 정밀 파싱
    match = re.match(r'^I_(.+)_([^_]+)_(\d+)$', img_name)
    if match:
        font_prefix, char_name, idx = match.groups()
    else:
        # 매칭 실패 시 기본 분할 방식으로 폴백 처리
        parts = img_name.split('_')
        char_name = parts[1] if len(parts) >= 2 else img_name

    safe_char_name = make_safe_filename(char_name)

    # 대상 타겟 이미지 경로는 [폰트명]_[글자].png 형식으로 target 디렉토리에 매핑되어 있습니다.
    target_img_name = f"{ttfname}_{char_name}"
    safe_target_img_name = make_safe_filename(target_img_name)

    input_path_val = f"{safe_img_name}.png"
    target_path_val = f"{safe_target_img_name}.png"

    # 딕셔너리에서 O(1)로 조회 후 인덱스 순 정렬
    sorted_input_strokes = sorted(input_strokes_dict.get(safe_img_name, []), key=lambda x: x[0])
    input_stroke_files = [item[1] for item in sorted_input_strokes]
    input_stroke_paths_val = ";".join(input_stroke_files)
    total_paths_val = len(input_stroke_files)

    sorted_target_strokes = sorted(target_strokes_dict.get(safe_target_img_name, []), key=lambda x: x[0])
    target_stroke_files = [item[1] for item in sorted_target_strokes]
    target_stroke_paths_val = ";".join(target_stroke_files)

    rows.append({
        "input_path": input_path_val,
        "target_path": target_path_val,
        "total_paths": total_paths_val,
        "input_stroke_paths": input_stroke_paths_val,
        "target_stroke_paths": target_stroke_paths_val
    })

# DataFrame 생성 및 병합 저장
df_new = pd.DataFrame(rows)
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

if os.path.exists(csv_path):
    print(f"기존 CSV 파일 존재 발견: {csv_path}. 신규 생성된 데이터를 우선하여 병합합니다.")
    try:
        df_existing = pd.read_csv(csv_path)
        df_results = pd.concat([df_new, df_existing], ignore_index=True).drop_duplicates(subset=["input_path"], keep="first")
    except Exception as e:
        print(f"기존 CSV 파일을 읽는 도중 오류 발생 ({e}). 새로 작성합니다.")
        df_results = df_new
else:
    print(f"CSV 파일이 없어 새로 생성합니다: {csv_path}")
    df_results = df_new

df_results.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"CSV 파일 업데이트 완료. 전체 데이터 수: {len(df_results)}개")
display(df_results.head(10))

In [ ]:
# @title 된거 안된거 구분하기

import os
import unicodedata
import pandas as pd

if not globals().get('inspection_completed', False):
    raise RuntimeError("미해결 검수가 남아있어 status 저장 및 삭제를 중단합니다.")

# 리스트 안의 모든 글자들을 NFC 형태로 정규화
# stat_path가 올바른 경로인지 최종 재정의 확인
stat_path = '/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/'+('test_dataset' if test else 'automated_train_dataset')+'/status/status' + ttfname + '.csv'

normalized_clean_list = [unicodedata.normalize('NFC', char_name) for char_name in local_target_clean]
normalized_skip_list = [unicodedata.normalize('NFC', char_name) for char_name in local_target_skip]

# 이번에 새로 작업한 타겟 전체 목록
norm_initial_work_chars = {unicodedata.normalize('NFC', c) for c in initial_work_chars}

# 1. 파일 삭제 범위 제한: 오직 이번에 실제 작업한 글자(initial_work_chars) 중 최종 skip으로 확정된 글자들만 정리
if os.path.exists(target_path):
    files = os.listdir(target_path)
    deleted_files = 0
    for file in files:
        if file.endswith('.png'):
            raw_char_name = os.path.splitext(file)[0]
            # ttfname 접두사가 포함된 실제 파일명에서 글자 부분 추출 또는 접두사 매칭 처리
            normalized_file_char = unicodedata.normalize('NFC', raw_char_name)

            # 접두사가 붙어있는 경우(예: NanumDaSiSiJagHae_아)와 원본(아) 모두 skip 리스트에 존재하는지 체크
            is_in_skip = (normalized_file_char in normalized_skip_list) or \
                         (any(normalized_file_char.endswith(s) for s in normalized_skip_list))

            if is_in_skip:
                file_path = os.path.join(target_path, file)
                try:
                    os.remove(file_path)
                    print(f"이미지 삭제됨: {file}")
                    deleted_files += 1
                except Exception as e:
                    print(f"{file} 이미지 삭제 실패: {e}")
    print(f"이미지 정리 완료: 이번 작업 대상 중 총 {deleted_files}개의 파일이 정리되었습니다.")
else:
    print(f"경로가 존재하지 않습니다: {target_path}")

# local_target_skip 리스트에 있는 문자들의 JSON 파일 삭제
if os.path.exists(bjson_path):
    json_files = os.listdir(bjson_path)
    deleted_jsons = 0
    for json_file in json_files:
        if json_file.endswith('.json'):
            raw_json_char = os.path.splitext(json_file)[0]
            normalized_json_char = unicodedata.normalize('NFC', raw_json_char)

            # 이번 작업 대상이었으면서, 최종적으로 skip 목록에 포함된 경우의 json만 삭제
            is_in_skip_json = (normalized_json_char in normalized_skip_list) or \
                              (any(normalized_json_char.endswith(s) for s in normalized_skip_list)) or \
                              (f"{ttfname}_{normalized_json_char}" in normalized_skip_list)

            if is_in_skip_json:
                json_file_path = os.path.join(bjson_path, json_file)
                try:
                    os.remove(json_file_path)
                    print(f"JSON 삭제됨: {json_file}")
                    deleted_jsons += 1
                except Exception as e:
                    print(f"{json_file} JSON 삭제 실패: {e}")
    print(f"JSON 정리 완료: 이번 작업 대상 중 총 {deleted_jsons}개의 파일이 정리되었습니다.")
else:
    print(f"경로가 존재하지 않습니다: {bjson_path}")

# 2. status.csv 불러오기 또는 새로 만들기
os.makedirs(os.path.dirname(stat_path), exist_ok=True)
if os.path.exists(stat_path):
    try:
        df_stat = pd.read_csv(stat_path)
    except Exception as e:
        print(f"CSV 로드 중 에러 발생, 새로 생성합니다: {e}")
        df_stat = pd.DataFrame(columns=['done', 'skip'])
else:
    df_stat = pd.DataFrame(columns=['done', 'skip'])

# 기존 값 불러와서 Set으로 정리
existing_done = set(df_stat['done'].dropna().astype(str).apply(lambda x: unicodedata.normalize('NFC', x)))
existing_skip = set(df_stat['skip'].dropna().astype(str).apply(lambda x: unicodedata.normalize('NFC', x)))

# 이번에 새로 clean으로 승인된 글자는 기존 skip 목록에서 안전하게 제거
for c in normalized_clean_list:
    existing_skip.discard(c)
    existing_done.add(c)

# 이번에 새로 skip으로 확정된 글자는 기존 done 목록에서 안전하게 제거
for s in normalized_skip_list:
    existing_done.discard(s)
    existing_skip.add(s)

updated_done = list(existing_done)
updated_skip = list(existing_skip)

# 길이를 맞춰서 DataFrame으로 다시 재조립
max_len = max(len(updated_done), len(updated_skip))

# 패딩 추가
updated_done += [None] * (max_len - len(updated_done))
updated_skip += [None] * (max_len - len(updated_skip))

df_updated = pd.DataFrame({
    'done': updated_done,
    'skip': updated_skip
})

# CSV 파일 저장
df_updated.to_csv(stat_path, index=False, encoding='utf-8-sig')
print(f"status.csv 업데이트 완료! (누적 완료: {len([x for x in updated_done if x is not None])}개, 누적 스킵: {len([x for x in updated_skip if x is not None])}개)")
print(stat_path)
display(df_updated.head(10))

# 드라이브에 올리기

In [ ]:
# @title
import os
import tarfile

# 구글 드라이브 대상 부모 디렉토리 설정
dest_parent_dir = "/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/automated_train_dataset"

# 압축 대상 서브폴더 리스트
subfolders = ['input', 'input_stroke', 'target', 'target_stroke']

print(f"대상 구글 드라이브 경로: {dest_parent_dir}")

for folder in subfolders:
    src_folder_path = os.path.join(output, folder)  # 로컬 소스 폴더 경로

    # 각 서브폴더를 위한 드라이브 하위 경로 생성 (예: .../automated_train_dataset/input/)
    drive_folder_dir = os.path.join(dest_parent_dir, folder)
    os.makedirs(drive_folder_dir, exist_ok=True)

    if os.path.exists(src_folder_path):
        # 기본 tar 파일명 설정 및 중복 검사 후 카운터 넘버링 적용
        counter = 1
        tar_filename = f"{folder}.tar"
        tar_output_path = os.path.join(drive_folder_dir, tar_filename)

        while os.path.exists(tar_output_path):
            tar_filename = f"{folder}_({counter}).tar"
            tar_output_path = os.path.join(drive_folder_dir, tar_filename)
            counter += 1

        print(f" -> 로컬 '{src_folder_path}' 폴더를 '{tar_output_path}'로 압축 및 드라이브 전송 시작...")

        # tar 파일 작성
        with tarfile.open(tar_output_path, "w") as tar:
            # 아카이브 내부에 폴더 구조가 중첩되지 않고 파일들만 담기도록 arcname을 설정하여 추가
            for filename in os.listdir(src_folder_path):
                file_path = os.path.join(src_folder_path, filename)
                if os.path.isfile(file_path):
                    tar.add(file_path, arcname=filename)

        print(f"    ✔ '{tar_filename}' 압축 완료!")
    else:
        print(f" -> 경고: 로컬 폴더 '{src_folder_path}'가 존재하지 않아 압축을 건너뜁니다.")

print("모든 개별 폴더의 .tar 압축 및 구글 드라이브 업로드가 완료되었습니다!")

In [ ]:
# @title 구글 드라이브에 원본 폴더 그대로 복사하기
import os
import shutil

# 구글 드라이브 대상 부모 디렉토리 설정
dest_parent_dir = "/content/drive/Shareddrives/2026 자율연구/HSQM/dataset/"+('test_dataset' if test else 'automated_train_dataset')

# 복사 대상 서브폴더 리스트
subfolders = ['input', 'input_stroke', 'target', 'target_stroke']

print(f"대상 구글 드라이브 경로: {dest_parent_dir}")

for folder in subfolders:
    src_folder_path = os.path.join(output, folder)  # 로컬 소스 폴더 경로
    drive_folder_dir = os.path.join(dest_parent_dir, folder)

    if os.path.exists(src_folder_path):
        print(f" -> 로컬 '{src_folder_path}' 폴더 안의 파일들을 '{drive_folder_dir}' 경로로 직접 전송 시작...")
        os.makedirs(drive_folder_dir, exist_ok=True)

        copied_count = 0
        for filename in os.listdir(src_folder_path):
            file_path = os.path.join(src_folder_path, filename)
            if os.path.isfile(file_path):
                dest_file_path = os.path.join(drive_folder_dir, filename)
                # 원본 파일 그대로 드라이브 경로에 복사 (이미 존재하면 덮어씀)
                shutil.copy2(file_path, dest_file_path)
                copied_count += 1

        print(f"    ✔ '{folder}' 폴더 내 파일 {copied_count}개 복사 완료!")
    else:
        print(f" -> 경고: 로컬 폴더 '{src_folder_path}'가 존재하지 않아 복사를 건너뜁니다.")

print("모든 개별 폴더의 원본 파일 구글 드라이브 동기화가 완료되었습니다!")

#초기화 시키기


In [ ]:
# @title
import shutil
import os

results_dir = '/content/results'
if os.path.exists(results_dir):
    shutil.rmtree(results_dir)
    print(f"디렉토리 '{results_dir}' 및 모든 내용이 삭제되었습니다.")
else:
    print(f"디렉토리 '{results_dir}'가 존재하지 않습니다.")


# 임시코드들

In [ ]:
import pandas as pd
import os

if os.path.exists(csv_path):
    df_csv = pd.read_csv(csv_path)
    # 각 행에서 결측치가 하나라도 존재하는지 확인
    null_rows_count = df_csv.isnull().any(axis=1).sum()
    print(f"CSV 파일 경로: {csv_path}")
    print(f"전체 행 개수: {len(df_csv)}개")
    print(f"결측치가 포함된 행 개수: {null_rows_count}개")
else:
    print(f"오류: {csv_path} 파일을 찾을 수 없습니다.")

In [ ]:
import pandas as pd
import os

if os.path.exists(csv_path):
    # 기존 CSV 로드
    df_csv = pd.read_csv(csv_path)

    # 결측치가 있는 행 제거 (모든 열 기준 하나라도 결측치가 있으면 삭제)
    df_cleaned = df_csv.dropna().reset_index(drop=True)

    # 정제된 데이터 다시 CSV로 저장 (utf-8-sig 인코딩 적용)
    df_cleaned.to_csv(csv_path, index=False, encoding='utf-8-sig')

    print(f"[업데이트 완료]")
    print(f"기존 행 개수: {len(df_csv)}개")
    print(f"결측치 제거 후 행 개수: {len(df_cleaned)}개")
    print(f"저장 경로: {csv_path}")
else:
    print(f"오류: {csv_path} 파일을 찾을 수 없습니다.")